# 🎵 SATB Arranger Tool - Google Colab

A comprehensive tool for choir arrangers that creates four-part SATB (Soprano, Alto, Tenor, Bass) arrangements from YouTube videos.

## Features
- Download audio from YouTube videos
- Extract and separate vocal stems using Spleeter
- Analyze harmonic content and create four-part arrangements
- Generate individual MIDI files for each voice part (S, A, T, B)
- Audio playback and visualization

**Note**: This notebook is designed to run on Google Colab with GPU acceleration for optimal performance.

## 🚀 Setup and Installation

First, we'll install all the required dependencies and set up the environment.

In [ ]:
# Install system dependencies
!apt-get update -qq
!apt-get install -y ffmpeg

print("✅ System dependencies installed successfully!")

In [ ]:
# Install Python dependencies
!pip install -q spleeter==2.3.2
!pip install -q yt-dlp==2023.12.30
!pip install -q pydub==0.25.1
!pip install -q numpy==1.24.3
!pip install -q scipy==1.10.1
!pip install -q librosa==0.10.1
!pip install -q midiutil==1.2.1
!pip install -q music21==9.1.0
!pip install -q tensorflow==2.13.0
!pip install -q ffmpeg-python==0.2.0
!pip install -q matplotlib==3.7.2
!pip install -q soundfile==0.12.1
!pip install -q ipython

print("✅ Python dependencies installed successfully!")

In [ ]:
# Setup project structure
import os
import sys

# Create project directory structure
directories = [
    'src',
    'output',
    'output/audio',
    'output/stems',
    'output/midi',
    'models'
]

for directory in directories:
    os.makedirs(directory, exist_ok=True)
    
print("✅ Project directory structure created!")
print("\n📁 Options for project setup:")
print("1. Upload your project files using the file browser on the left")
print("2. Clone from GitHub (modify the cell below)")
print("3. Use the demo implementation provided in this notebook")

## 📁 Project Files Setup

Choose one of the following options:

### Option 1: Clone from GitHub (Recommended)
If your project is on GitHub, uncomment and modify the next cell:

In [ ]:
# Option 1: Clone from your GitHub repository
# Uncomment and modify the following lines:

# !git clone https://github.com/YOUR_USERNAME/YOUR_REPO_NAME.git
# os.chdir('YOUR_REPO_NAME')
# !cp -r src/* ../src/
# os.chdir('..')

print("Option 1: To use this, uncomment and modify the git clone command above")

### Option 2: Manual File Upload
Upload your project files using Colab's file browser on the left panel.

### Option 3: Demo Implementation
Use the simplified demo version provided below:

In [ ]:
# Create demo YouTube Downloader
youtube_downloader_code = '''import yt_dlp
import os
from pathlib import Path

class YouTubeDownloader:
    def __init__(self, output_dir="output/audio"):
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(parents=True, exist_ok=True)
    
    def download_audio(self, url, format="mp3"):
        """Download audio from YouTube URL."""
        ydl_opts = {
            'format': 'bestaudio/best',
            'outtmpl': str(self.output_dir / '%(title)s.%(ext)s'),
            'postprocessors': [{
                'key': 'FFmpegExtractAudio',
                'preferredcodec': format,
                'preferredquality': '192',
            }],
        }
        
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            info = ydl.extract_info(url, download=True)
            title = info.get('title', 'audio')
            
        # Find the downloaded file
        for file in self.output_dir.glob(f"*{format}"):
            return str(file)
        
        raise FileNotFoundError("Could not find downloaded audio file")
'''

with open('src/youtube_downloader.py', 'w') as f:
    f.write(youtube_downloader_code)

print("✅ YouTube downloader module created!")

In [ ]:
# Create demo Stem Separator
stem_separator_code = '''from spleeter.separator import Separator
import librosa
import numpy as np
import soundfile as sf
from pathlib import Path

class StemSeparator:
    def __init__(self, output_dir="output/stems"):
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(parents=True, exist_ok=True)
        
    def separate_vocals(self, audio_path, stems=2):
        """Separate vocals from accompaniment using Spleeter."""
        separator = Separator(f'spleeter:{stems}stems')
        
        # Load audio
        waveform, sample_rate = librosa.load(audio_path, sr=44100, mono=False)
        if waveform.ndim == 1:
            waveform = np.stack([waveform, waveform])
        
        # Separate
        prediction = separator.separate(waveform.T)
        
        # Save stems
        stems_dict = {}
        for stem_name, stem_data in prediction.items():
            output_path = self.output_dir / f"{Path(audio_path).stem}_{stem_name}.wav"
            sf.write(str(output_path), stem_data, sample_rate)
            stems_dict[stem_name] = str(output_path)
            
        return stems_dict
    
    def extract_vocal_melody(self, vocal_path):
        """Extract melody information from vocal track."""
        y, sr = librosa.load(vocal_path)
        
        # Extract tempo
        tempo, _ = librosa.beat.beat_track(y=y, sr=sr)
        
        # Simple key estimation
        chroma = librosa.feature.chroma_stft(y=y, sr=sr)
        key_profile = np.mean(chroma, axis=1)
        key_index = np.argmax(key_profile)
        keys = ['C', 'C#', 'D', 'D#', 'E', 'F', 'F#', 'G', 'G#', 'A', 'A#', 'B']
        estimated_key = keys[key_index]
        
        return {
            'tempo': float(tempo),
            'key': estimated_key,
            'sample_rate': sr,
            'audio_data': y
        }
'''

with open('src/stem_separator.py', 'w') as f:
    f.write(stem_separator_code)

print("✅ Stem separator module created!")

In [ ]:
# Create demo SATB Arranger
arranger_code = '''import numpy as np
import librosa

class SATBArranger:
    def __init__(self):
        self.voice_ranges = {
            'soprano': {'min': 60, 'max': 84},  # C4 to C6
            'alto': {'min': 55, 'max': 77},     # G3 to F5
            'tenor': {'min': 48, 'max': 69},    # C3 to A4
            'bass': {'min': 40, 'max': 62}      # E2 to D4
        }
    
    def create_arrangement(self, melody_data, arrangement_style='hymn'):
        """Create SATB arrangement from melody data."""
        # Extract melody notes (simplified)
        y = melody_data['audio_data']
        sr = melody_data['sample_rate']
        
        # Extract pitch using librosa
        pitches, magnitudes = librosa.piptrack(y=y, sr=sr, threshold=0.1)
        
        # Get the most prominent pitch per frame
        melody_notes = []
        for t in range(pitches.shape[1]):
            index = magnitudes[:, t].argmax()
            pitch = pitches[index, t]
            if pitch > 0:
                midi_note = librosa.hz_to_midi(pitch)
                melody_notes.append(int(midi_note))
        
        # Filter out invalid notes and create a simplified melody
        melody_notes = [note for note in melody_notes if 40 <= note <= 85]
        if not melody_notes:
            # Fallback melody in C major
            melody_notes = [60, 62, 64, 65, 67, 69, 71, 72]  # C major scale
        
        # Take a representative sample of notes
        if len(melody_notes) > 16:
            step = len(melody_notes) // 16
            melody_notes = melody_notes[::step][:16]
        
        # Create harmonies based on style
        if arrangement_style == 'hymn':
            return self._create_hymn_style(melody_notes)
        else:
            return self._create_basic_harmony(melody_notes)
    
    def _create_hymn_style(self, melody_notes):
        """Create hymn-style four-part harmony."""
        soprano = melody_notes
        
        # Alto: typically a third or fourth below soprano
        alto = [max(note - 3, self.voice_ranges['alto']['min']) for note in soprano]
        
        # Tenor: often in close harmony with alto
        tenor = [max(note - 7, self.voice_ranges['tenor']['min']) for note in soprano]
        
        # Bass: root notes and harmonic foundation
        bass = [max(note - 12, self.voice_ranges['bass']['min']) for note in soprano]
        
        return {
            'soprano': soprano,
            'alto': alto,
            'tenor': tenor,
            'bass': bass,
            'tempo': 120,
            'time_signature': (4, 4)
        }
    
    def _create_basic_harmony(self, melody_notes):
        """Create basic four-part harmony."""
        return self._create_hymn_style(melody_notes)
'''

with open('src/arranger.py', 'w') as f:
    f.write(arranger_code)

print("✅ SATB arranger module created!")

In [ ]:
# Create demo MIDI Generator
midi_generator_code = '''from midiutil import MIDIFile
from pathlib import Path

class MIDIGenerator:
    def __init__(self, output_dir="output/midi"):
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(parents=True, exist_ok=True)
        self.instruments = {
            'soprano': 52,  # Choir Aahs
            'alto': 52,
            'tenor': 52,
            'bass': 52
        }
        
    def set_midi_instruments(self, parts):
        """Set appropriate MIDI instruments for each voice."""
        pass  # Instruments already set in __init__
        
    def export_parts(self, parts):
        """Export individual MIDI files for each voice part."""
        midi_paths = {}
        
        for voice, notes in parts.items():
            if voice in ['soprano', 'alto', 'tenor', 'bass']:
                midi_file = MIDIFile(1)
                track = 0
                channel = 0
                tempo = parts.get('tempo', 120)
                
                midi_file.addTempo(track, 0, tempo)
                midi_file.addProgramChange(track, channel, 0, self.instruments.get(voice, 52))
                
                # Add notes
                time = 0
                duration = 1  # Quarter note
                volume = 100
                
                for note in notes:
                    midi_file.addNote(track, channel, note, time, duration, volume)
                    time += duration
                
                # Save file
                output_path = self.output_dir / f"{voice}.mid"
                with open(output_path, "wb") as output_file:
                    midi_file.writeFile(output_file)
                
                midi_paths[voice] = str(output_path)
        
        return midi_paths
    
    def export_combined(self, parts):
        """Export a combined MIDI file with all voice parts."""
        midi_file = MIDIFile(4)  # 4 tracks
        tempo = parts.get('tempo', 120)
        
        voices = ['soprano', 'alto', 'tenor', 'bass']
        
        for track, voice in enumerate(voices):
            midi_file.addTempo(track, 0, tempo)
            midi_file.addProgramChange(track, track, 0, self.instruments.get(voice, 52))
            midi_file.addTrackName(track, 0, voice.capitalize())
            
            # Add notes
            time = 0
            duration = 1
            volume = 100
            
            for note in parts[voice]:
                midi_file.addNote(track, track, note, time, duration, volume)
                time += duration
        
        # Save combined file
        output_path = self.output_dir / "satb_arrangement.mid"
        with open(output_path, "wb") as output_file:
            midi_file.writeFile(output_file)
        
        return str(output_path)
'''

with open('src/midi_generator.py', 'w') as f:
    f.write(midi_generator_code)

# Create __init__.py
with open('src/__init__.py', 'w') as f:
    f.write('# SATB Arranger Package\n')

print("✅ MIDI generator module created!")
print("✅ All demo modules created successfully!")

## 🤖 Download Spleeter Models

Spleeter needs to download pre-trained models for audio separation.

In [ ]:
# Download Spleeter models
print("Downloading Spleeter models... This may take a few minutes.")

from spleeter.separator import Separator

# Download 2-stem model (vocals/accompaniment)
separator_2stems = Separator('spleeter:2stems')
print("✅ 2-stem model downloaded!")

print("🎉 Spleeter setup complete!")

## 🎵 Using the SATB Arranger

Now let's demonstrate how to use the tool with a YouTube video.

In [ ]:
# Import our modules
import sys
sys.path.append('src')

from youtube_downloader import YouTubeDownloader
from stem_separator import StemSeparator
from arranger import SATBArranger
from midi_generator import MIDIGenerator

# Initialize components
downloader = YouTubeDownloader()
separator = StemSeparator()
arranger = SATBArranger()
midi_gen = MIDIGenerator()

print("✅ All modules imported successfully!")

In [ ]:
# Process a YouTube video
# Replace with your desired YouTube URL
youtube_url = "https://www.youtube.com/watch?v=dQw4w9WgXcQ"  # Example URL - change this!

print(f"Processing: {youtube_url}")
print("\n" + "="*50)

try:
    # Step 1: Download audio
    print("\n1. 📥 Downloading audio from YouTube...")
    audio_path = downloader.download_audio(youtube_url)
    print(f"   ✅ Audio saved to: {audio_path}")
    
    # Step 2: Separate stems
    print("\n2. 🎤 Separating vocals from accompaniment...")
    stems = separator.separate_vocals(audio_path, stems=2)
    vocal_path = stems['vocals']
    print(f"   ✅ Vocals extracted to: {vocal_path}")
    
    # Step 3: Analyze melody
    print("\n3. 🎼 Analyzing vocal melody...")
    melody_data = separator.extract_vocal_melody(vocal_path)
    print(f"   ✅ Tempo: {melody_data['tempo']:.0f} BPM")
    print(f"   ✅ Key: {melody_data['key']}")
    
    # Step 4: Create arrangement
    print("\n4. 🎹 Creating SATB arrangement...")
    parts = arranger.create_arrangement(melody_data, arrangement_style='hymn')
    print("   ✅ Four-part harmony created")
    print(f"   📊 Soprano notes: {len(parts['soprano'])}")
    print(f"   📊 Alto notes: {len(parts['alto'])}")
    print(f"   📊 Tenor notes: {len(parts['tenor'])}")
    print(f"   📊 Bass notes: {len(parts['bass'])}")
    
    # Step 5: Export MIDI
    print("\n5. 💾 Exporting MIDI files...")
    midi_gen.set_midi_instruments(parts)
    midi_paths = midi_gen.export_parts(parts)
    combined_path = midi_gen.export_combined(parts)
    
    print("\n🎉 Success! Files created:")
    for voice, path in midi_paths.items():
        print(f"   🎵 {voice.capitalize()}: {path}")
    print(f"   🎼 Combined: {combined_path}")
    
except Exception as e:
    print(f"\n❌ Error: {str(e)}")
    import traceback
    traceback.print_exc()

## 📊 Visualization and Analysis

Let's visualize the generated SATB arrangement.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Visualize the SATB arrangement
if 'parts' in locals():
    fig, ax = plt.subplots(figsize=(12, 8))
    
    voices = ['soprano', 'alto', 'tenor', 'bass']
    colors = ['red', 'orange', 'blue', 'green']
    
    for i, voice in enumerate(voices):
        notes = parts[voice]
        x = range(len(notes))
        ax.plot(x, notes, 'o-', color=colors[i], label=voice.capitalize(), linewidth=2, markersize=6)
    
    ax.set_xlabel('Note Position')
    ax.set_ylabel('MIDI Note Number')
    ax.set_title('SATB Arrangement Visualization')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Add note names on y-axis
    note_names = ['C', 'C#', 'D', 'D#', 'E', 'F', 'F#', 'G', 'G#', 'A', 'A#', 'B']
    
    # Create custom y-tick labels
    y_min, y_max = ax.get_ylim()
    y_ticks = range(int(y_min), int(y_max) + 1, 12)  # Every octave
    y_labels = [f"{note_names[tick % 12]}{tick // 12 - 1}" for tick in y_ticks]
    ax.set_yticks(y_ticks)
    ax.set_yticklabels(y_labels)
    
    plt.tight_layout()
    plt.show()
    
    print("\n📈 Arrangement Analysis:")
    for voice in voices:
        notes = parts[voice]
        print(f"   {voice.capitalize():8s}: Range {min(notes):3d}-{max(notes):3d}, Avg: {np.mean(notes):.1f}")
else:
    print("No arrangement data available. Please run the processing cell first.")

## 🔊 Audio Playback

Listen to the original audio and separated vocals.

In [ ]:
from IPython.display import Audio, display
import os

# Play the separated vocals if available
if 'vocal_path' in locals() and os.path.exists(vocal_path):
    print("🎤 Separated Vocals:")
    display(Audio(vocal_path))
    
    # Also play accompaniment if available
    if 'stems' in locals() and 'accompaniment' in stems:
        accompaniment_path = stems['accompaniment']
        if os.path.exists(accompaniment_path):
            print("\n🎼 Accompaniment:")
            display(Audio(accompaniment_path))
else:
    print("No audio files available. Please run the processing cell first.")

## 📁 Download Generated Files

Download the generated MIDI files to your computer.

In [ ]:
from google.colab import files
import os
import zipfile

# Create a zip file with all MIDI files
if 'midi_paths' in locals():
    zip_filename = 'satb_arrangement.zip'
    
    with zipfile.ZipFile(zip_filename, 'w') as zipf:
        # Add individual voice files
        for voice, path in midi_paths.items():
            if os.path.exists(path):
                zipf.write(path, f"{voice}.mid")
        
        # Add combined file
        if 'combined_path' in locals() and os.path.exists(combined_path):
            zipf.write(combined_path, "satb_arrangement.mid")
    
    print(f"📦 Created zip file: {zip_filename}")
    
    # Download the zip file
    files.download(zip_filename)
    print("⬇️ Download started!")
    
else:
    print("No MIDI files available. Please run the processing cell first.")

## 🚀 Advanced Usage

Try different arrangement styles and settings.

In [ ]:
# Try different arrangement styles
if 'melody_data' in locals():
    styles = ['hymn', 'jazz', 'classical']
    
    for style in styles:
        print(f"\n🎵 Creating {style.upper()} arrangement...")
        
        try:
            parts_style = arranger.create_arrangement(melody_data, arrangement_style=style)
            
            # Export MIDI for this style
            midi_gen.set_midi_instruments(parts_style)
            style_midi_paths = midi_gen.export_parts(parts_style)
            
            # Rename files to include style
            for voice, path in style_midi_paths.items():
                new_path = path.replace('.mid', f'_{style}.mid')
                os.rename(path, new_path)
                print(f"   ✅ {voice.capitalize()} ({style}): {new_path}")
                
        except Exception as e:
            print(f"   ❌ Error creating {style} arrangement: {e}")
else:
    print("No melody data available. Please run the processing cell first.")

## 💡 Tips and Troubleshooting

### For best results:
1. **Choose clear vocal recordings** - Songs with prominent, clear vocals work best
2. **Avoid heavily processed songs** - Auto-tuned or heavily processed vocals may not separate well
3. **Try shorter clips** - For testing, use shorter YouTube videos (1-3 minutes)
4. **Check GPU usage** - Enable GPU in Colab (Runtime → Change runtime type → GPU) for faster processing

### Common issues:
- **Out of memory**: Try shorter audio clips or restart runtime
- **Download fails**: Check if the YouTube URL is valid and accessible
- **Poor separation**: Some songs don't separate well - try different videos

### Customization:
- Modify the `voice_ranges` in the arranger for different vocal ranges
- Adjust the harmony logic in the arranger for different styles
- Change MIDI instruments in the generator for different sounds

## 🎯 Next Steps

To extend this notebook:
1. **Upload your full project files** to replace the demo implementations
2. **Add music notation export** using music21
3. **Implement more arrangement styles**
4. **Add harmonic analysis visualization**
5. **Create a simple web interface** using Streamlit (note: Streamlit doesn't run directly in Colab)

For production use, consider:
- Setting up the full environment locally or on a cloud server
- Using the Streamlit web interface for easier use
- Adding more sophisticated music theory rules
- Implementing better audio processing algorithms

---

**🎵 Happy arranging! 🎵**